# 07 — XGBoost, Boosting (Stage 9)
`objective='multi:softprob'` (multi-class with probabilities). Class weighting is tuned through balanced sample weights (`XGBClassifierCW` in `src/models.py`).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
FIG = ROOT / 'figures'; REP = ROOT / 'reports'

In [2]:
summary = pd.read_csv(REP / 'tuning' / 'tuning_summary.csv').set_index('model')
row = summary.loc['xgboost']
print('Search             :', row['search'], f"({row['n_candidates']} candidates)")
print('CV Macro F1        :', f"{row['cv_macro_f1_mean']:.4f} ± {row['cv_macro_f1_std']:.4f}")
print('Train Macro F1     :', f"{row['train_macro_f1_mean']:.4f}")
print('Train - CV gap     :', f"{row['train_minus_cv_gap']:+.4f}")
print('Best parameters    :', row['best_params'])

Search             : RandomizedSearchCV (30 candidates)
CV Macro F1        : 0.7956 ± 0.0294
Train Macro F1     : 0.9936
Train - CV gap     : +0.1980
Best parameters    : {'model__class_weight': 'balanced', 'model__colsample_bytree': 0.8253152871382157, 'model__learning_rate': 0.05679563707774164, 'model__max_depth': 4, 'model__min_child_weight': 1, 'model__n_estimators': 489, 'model__subsample': 0.6964101864104046}


Fold-wise Macro F1 (same 5 stratified folds for every model):

In [3]:
folds = pd.read_csv(REP / 'tuning' / 'cv_fold_scores.csv', index_col=0)
folds.loc[[i for i in folds.index if i.startswith('XGBoost')]]

,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
XGBoost (initial),0.7708,0.7153,0.7676,0.7911,0.7651,0.7620,0.0251
XGBoost,0.8210,0.7978,0.7823,0.8295,0.7472,0.7956,0.0294


Top 10 candidates of the search, ranked by mean CV Macro F1:

In [4]:
res = pd.read_csv(REP / 'tuning' / 'xgboost_search_results.csv')
cols = [c for c in res.columns if c.startswith('param_')] + ['mean_test_score', 'std_test_score', 'mean_train_score', 'mean_fit_time']
res.sort_values('rank_test_score')[cols].head(10).round(4)

,param_model__class_weight,param_model__colsample_bytree,param_model__learning_rate,param_model__max_depth,param_model__min_child_weight,param_model__n_estimators,param_model__subsample,mean_test_score,std_test_score,mean_train_score,mean_fit_time
6,balanced,0.8253,0.0568,4,1,489,0.6964,0.7956,0.0294,0.9936,1.1708
8,balanced,0.9021,0.0632,8,2,626,0.9878,0.7932,0.0333,0.9988,1.2103
22,balanced,0.6440,0.0371,8,1,408,0.6028,0.7915,0.0338,0.9737,1.2046
9,balanced,0.7799,0.0583,6,2,420,0.6354,0.7889,0.0319,0.9659,1.0126
7,balanced,0.6488,0.0765,5,2,230,0.7035,0.7882,0.0350,0.9467,0.5751
18,balanced,0.8546,0.0469,6,2,634,0.6997,0.7872,0.0379,0.9847,1.5445
5,NaN,0.9440,0.1262,3,3,423,0.9796,0.7830,0.0285,1.0000,0.7422
4,balanced,0.9895,0.0376,8,3,393,0.8370,0.7807,0.0297,0.9316,0.9889
13,NaN,0.8827,0.1440,3,3,639,0.7434,0.7783,0.0321,1.0000,1.1796
2,balanced,0.9754,0.0200,6,1,310,0.7217,0.7777,0.0153,0.8573,0.8566
